<img src="https://raw.githubusercontent.com/claudiogonzaga/akoe/main/akoe.png" width="120" align="left" hspace="16" vspace="4">

# Akoé

*Do grego ἀκοή — «escuta».*

<br clear="left">

### Transcreve todos os áudios e vídeos de uma pasta do Google Drive e consolida tudo em um único Google Doc, usando o Whisper (OpenAI).

*Requer autorizar o acesso ao seu Google Drive. Só você tem acesso ao conteúdo do seu Drive.*

**Como funciona:**
1. Você cola o link de uma pasta do Drive (ou envia arquivos por upload).
2. Cada áudio/vídeo da pasta é transcrito; arquivos longos são fragmentados automaticamente para não estourar a memória.
3. As transcrições vão para um único Google Doc criado na própria pasta, com um sumário no topo que marca o que já foi feito.
4. Se você rodar de novo, só os arquivos ainda não transcritos são processados.

### **Para um desempenho mais rápido, configure o ambiente de execução para "GPU"**
*Menu "Ambiente de execução" → "Alterar o tipo de ambiente de execução" → selecione "GPU".*

**Ao final, o log de progresso é apagado e fica apenas o link do documento.**

In [ ]:
# -*- coding: utf-8 -*-
"""Akoé — Transcrever Mídias de Uma Pasta do Drive

###Esta aplicação extrairá áudio de todos os arquivos de vídeo em uma pasta do Google Drive e criará uma transcrição de alta qualidade com o sistema de reconhecimento automático de fala Whisper da OpenAI.

*Nota: Isso requer dar permissão à aplicação para se conectar ao seu drive. Apenas você terá acesso ao conteúdo do seu drive.*

Esta aplicação:
1. Conecta-se ao seu Google Drive quando você dá permissão.
2. Cria uma pasta WhisperVideo e 2 subpastas.
3. Quando você executa a aplicação, ela procurará por todos os arquivos de vídeo ou áudio na pasta principal, transcreverá e depois moverá o arquivo processado para a pasta de arquivos processados e a transcrição na pasta de textos das transcrições.

###**Para um desempenho mais rápido, configure seu ambiente de execução para "GPU"**
*Clique em "Runtime" no menu e clique em "Alterar tipo de ambiente de execução". Selecione "GPU".*

**Nota: Se você adicionar um novo arquivo após executar esta aplicação, será necessário remontar o drive na etapa 1 para torná-los pesquisáveis**
"""

# @title Licensed under the Apache License, Version 2.0 (the "License");
# Você não pode usar este arquivo exceto em conformidade com a Licença.
# Você pode obter uma cópia da Licença em
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# A menos que exigido pela lei aplicável ou acordado por escrito, o software
# distribuído sob a Licença é distribuído "COMO ESTÁ",
# SEM GARANTIAS OU CONDIÇÕES DE QUALQUER TIPO, expressas ou implícitas.
# Consulte a Licença para o texto específico que rege as permissões e
# limitações sob a Licença.

# Configuração
!pip install -U -q "openai-whisper" transformers torch accelerate

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import os
import re
import time
import subprocess
import textwrap

# Reduz a fragmentação de memória da GPU — recomendado pelo PyTorch para
# evitar erros de "CUDA out of memory". Precisa estar definido antes de
# importar o torch (que é importado mais abaixo, em setup_whisper).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Autenticação do usuário (sem persistir credenciais em disco)
from google.colab import drive, auth
import google.auth

drive.mount('/content/drive')

auth.authenticate_user()
creds, _ = google.auth.default()

# Analisar os argumentos
modelo_whisper = 'large-v3'  # @param ["tiny", "base", "small", "medium", "large", "large-v2", "large-v3", "pierreguillou/whisper-medium-portuguese", "whisperx-large-v3 (experimental)"] {allow-input: true}

# Link da pasta no Google Drive com os áudios/vídeos.
# O modo de entrada é automático: com um link aqui, lê os arquivos dessa pasta
# do Drive; em branco, abre o seletor de upload do seu computador (nesse caso
# nada toca o Drive, e a transcrição é baixada de volta ao final).
# Deixe em branco no notebook do GitHub: cole o link só na hora de usar.
PASTA_DOCUMENTOS = ""  # @param {type:"string"}

USAR_DRIVE = bool(PASTA_DOCUMENTOS.strip())

# O que fazer com os arquivos depois de transcrever.
# "mídia original" = o áudio/vídeo que estava na pasta do Drive.
# "áudio extraído" = o WAV 16 kHz mono gerado a partir de vídeos; apagá-lo
# significa não guardar cópia na subpasta "Áudios Extraídos".
ACAO_ARQUIVOS = "Manter mídia original e apagar áudio eventualmente extraído"  # @param ["Manter mídia original e apagar áudio eventualmente extraído", "Apagar mídia original e manter áudio eventualmente extraído", "Apagar tudo depois de transcrever", "Manter tudo depois de transcrever"]

_ACOES = {
    # opção do dropdown: (apagar a mídia original, guardar o áudio extraído)
    "Manter mídia original e apagar áudio eventualmente extraído": (False, False),
    "Apagar mídia original e manter áudio eventualmente extraído": (True, True),
    "Apagar tudo depois de transcrever": (True, False),
    "Manter tudo depois de transcrever": (False, True),
}
APAGAR_ORIGINAL, GUARDAR_AUDIOS = _ACOES[ACAO_ARQUIVOS]

# Comando para transcrição de mídias
COMANDO_TRANSCRITOR = textwrap.dedent("""# Transcrever arquivos de áudio e vídeo
Você é um Transcritor Jurídico especializado em audiências e depoimentos. Sua tarefa é transcrever fielmente os arquivos de áudio e vídeo fornecidos, respeitando a fala de cada participante (ex: Promotor, Réu, Testemunha, Juiz). Não resuma, não interprete, apenas transcreva integralmente o conteúdo falado. Identifique os interlocutores sempre que possível. O texto deve ser claro, com pontuação adequada, mantendo a fidelidade ao original. Jamais invente falas ou preencha lacunas com suposições. Se houver trechos ininteligíveis, indique como [áudio ininteligível].""")

# --- Motor de transcrição selecionado ---
# WhisperX (experimental): mesmo Whisper por baixo, com detecção de voz, lote e
# tempos por palavra. É mais rápido e tem tempos melhores, mas é sensível às
# versões de torch/CUDA da Colab — pode falhar ou exigir reiniciar o ambiente.
IS_WHISPERX = modelo_whisper.strip().lower().startswith("whisperx")
IS_HF_MODEL = (not IS_WHISPERX) and "/" in modelo_whisper

# Tamanho do Whisper carregado quando o WhisperX está selecionado
MODELO_WHISPERX = "large-v3"
WHISPERX_BATCH_SIZE = 8

if IS_WHISPERX:
    print("⚠️  WhisperX é EXPERIMENTAL: instalando (pode demorar alguns minutos)…")
    print("    Se falhar, escolha um modelo comum (ex.: 'large-v3') e rode de novo.")
    subprocess.run(["pip", "install", "-q", "whisperx"], check=False)

# --- Carimbo de tempo no texto final ---
# Agrupa a transcrição em blocos de duração fixa, cada um marcado pelo tempo em
# que o bloco começa no arquivo: [00:00:00], [00:03:00], [00:06:00]...
# "Por segmento do modelo" usa os cortes do próprio Whisper (irregulares, de
# poucos segundos cada). Os tempos já vêm do modelo: não há custo extra.
CARIMBO_TEMPO = "A cada 3 minutos"  # @param ["A cada 1 minuto", "A cada 2 minutos", "A cada 3 minutos", "A cada 5 minutos", "A cada 10 minutos", "Por segmento do modelo", "Sem carimbo de tempo"]

_INTERVALOS = {
    "A cada 1 minuto": 60,
    "A cada 2 minutos": 120,
    "A cada 3 minutos": 180,
    "A cada 5 minutos": 300,
    "A cada 10 minutos": 600,
    "Por segmento do modelo": 0,     # um carimbo por segmento do modelo
    "Sem carimbo de tempo": None,    # texto corrido, sem tempos
}
INTERVALO_CARIMBO_S = _INTERVALOS[CARIMBO_TEMPO]

# --- Tag do modelo para o nome dos documentos ---
MODEL_TAG = f"[{modelo_whisper.split('/')[-1].replace(' (experimental)', '')}]"

# --- Prefixo do documento consolidado ---
PREFIXO_CONSOLIDADO = f"{MODEL_TAG} Transcrições de"

# --- Marcadores usados no documento consolidado para identificar seções ---
MARCADOR_INICIO = "═══ TRANSCRIÇÃO: "
MARCADOR_FIM = "═══ FIM DA TRANSCRIÇÃO: "
MARCADOR_SUMARIO = "══════ SUMÁRIO ══════"


# --- Privacidade das saídas impressas -----------------------------------
# As saídas ficam salvas dentro do .ipynb. Como este notebook mora num
# repositório público, os nomes reais dos arquivos e o ID da pasta são
# mascarados no que é IMPRESSO. O documento consolidado no Drive continua
# com os nomes completos — a máscara vale só para o log na tela.
MASCARAR_SAIDAS = True


def limpar_log():
    """Apaga todo o log impresso até aqui, deixando só o que for impresso
    depois. O documento consolidado já está salvo no Drive, então o log de
    progresso não tem mais utilidade — e é ele que carrega nomes de arquivos
    e IDs de pasta para dentro do .ipynb salvo."""
    try:
        from IPython.display import clear_output
        clear_output(wait=True)
    except Exception:
        pass


def mascarar(texto, inicio=3, fim=2):
    """Mascara o miolo de um texto: 'TAC concurso público.mp4' → 'TAC…p4'."""
    if not MASCARAR_SAIDAS or texto is None:
        return texto
    texto = str(texto)
    if len(texto) <= inicio + fim:
        return "…"
    return f"{texto[:inicio]}{'…' * 3}{texto[-fim:]}"

# --- Fragmentação para arquivos longos (evita OOM em arquivos muito grandes) ---
# Se a duração ultrapassar LIMITE_DURACAO_S, o arquivo é dividido em pedaços
# de CHUNK_DURACAO_S e cada pedaço é transcrito separadamente.
LIMITE_DURACAO_S = 1200  # 20 minutos
CHUNK_DURACAO_S = 600    # 10 minutos por fragmento


def _liberar_memoria_gpu():
    """Libera a memória residual da GPU deixada por execuções anteriores."""
    import gc
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


def setup_whisper():
    # Em re-execuções no mesmo kernel da Colab (ex.: "Drive already mounted"),
    # um modelo carregado antes pode continuar ocupando toda a VRAM e causar
    # "CUDA out of memory" ao carregar o novo. Removemos qualquer modelo global
    # anterior e limpamos o cache ANTES de carregar.
    global model
    try:
        del model
    except NameError:
        pass
    _liberar_memoria_gpu()

    if IS_WHISPERX:
        import whisperx
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        compute_type = "float16" if device == "cuda" else "int8"
        print(f"🔧 Carregando WhisperX ({MODELO_WHISPERX}) — experimental")
        print(f"   Dispositivo: {'GPU (CUDA)' if device == 'cuda' else 'CPU'}")
        return whisperx.load_model(
            MODELO_WHISPERX, device, compute_type=compute_type, language="pt"
        )

    if IS_HF_MODEL:
        from transformers import pipeline
        import torch
        device = 0 if torch.cuda.is_available() else "cpu"
        print(f"🔧 Carregando modelo HuggingFace: {modelo_whisper}")
        print(f"   Dispositivo: {'GPU (CUDA)' if device == 0 else 'CPU'}")
        pipe = pipeline(
            task="automatic-speech-recognition",
            model=modelo_whisper,
            chunk_length_s=30,
            device=device,
            ignore_warning=True,
        )
        pipe.model.config.forced_decoder_ids = (
            pipe.tokenizer.get_decoder_prompt_ids(language="pt", task="transcribe")
        )
        return pipe
    else:
        import whisper
        try:
            return whisper.load_model(modelo_whisper)
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            # Mesmo após a limpeza não há VRAM suficiente: cai para CPU em vez de
            # quebrar (mais lento, mas conclui). Para voltar à GPU, use o menu
            # "Ambiente de execução" → "Desconectar e excluir ambiente de execução"
            # para liberar a GPU, ou escolha um modelo menor (ex.: "medium").
            print(f"⚠️ Sem VRAM suficiente para o modelo '{modelo_whisper}'. "
                  "Carregando na CPU (mais lento).")
            _liberar_memoria_gpu()
            return whisper.load_model(modelo_whisper, device="cpu")

def extract_audio(video_path):
    """Extrai áudio de um arquivo de vídeo (ou áudio) para WAV 16 kHz mono via ffmpeg.

    Aceita qualquer formato de entrada. Se a entrada já for um .wav com o mesmo
    nome do destino, evita a colisão input==output gerando um sufixo distinto.
    """
    audio_path = video_path.rsplit(".", 1)[0] + ".wav"
    if audio_path == video_path:
        audio_path = video_path.rsplit(".", 1)[0] + "_extracted.wav"
    subprocess.run(
        ["ffmpeg", "-y", "-i", video_path, "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", audio_path],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )
    return audio_path


def obter_duracao_segundos(path):
    """Retorna a duração de um arquivo de áudio ou vídeo em segundos via ffprobe."""
    try:
        result = subprocess.run(
            ["ffprobe", "-v", "error", "-show_entries", "format=duration",
             "-of", "default=noprint_wrappers=1:nokey=1", path],
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            check=True,
            text=True,
        )
        return float(result.stdout.strip())
    except Exception:
        return 0.0


def fragmentar_audio(audio_path, chunk_s):
    """Fragmenta um WAV em pedaços de chunk_s segundos. Retorna lista de paths em ordem."""
    import glob
    base = audio_path.rsplit(".", 1)[0]
    pattern = f"{base}_chunk_%03d.wav"
    subprocess.run(
        ["ffmpeg", "-y", "-i", audio_path,
         "-f", "segment", "-segment_time", str(chunk_s),
         "-c:a", "pcm_s16le", "-ar", "16000", "-ac", "1",
         pattern],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )
    return sorted(glob.glob(f"{base}_chunk_*.wav"))


# Inicialização do modelo
model = setup_whisper()


def get_documents_from_drive(drive_folder_link):
    """Obtém arquivos de mídia da pasta do Google Drive e identifica documento consolidado existente."""
    print("\nIniciando leitura dos documentos na pasta do Drive...")
    MEDIA_MIME_TYPES = ['video/', 'audio/']

    try:
        folder_id = drive_folder_link.split('folders/')[-1].split('?')[0]
        print(f"ID da pasta: {mascarar(folder_id)}")
        service = build('drive', 'v3', cache_discovery=False, credentials=creds)

        # Listar todos os arquivos da pasta
        items = []
        page_token = None
        while True:
            response = service.files().list(
                q=f"'{folder_id}' in parents and trashed = false and mimeType != 'application/vnd.google-apps.shortcut'",
                fields="nextPageToken, files(id, name, mimeType)",
                pageSize=1000,
                pageToken=page_token
            ).execute()
            items.extend(response.get('files', []))
            page_token = response.get('nextPageToken', None)
            if not page_token:
                break

        if not items:
            print('Nenhum arquivo encontrado na pasta.')
            return {}, folder_id, None

        print(f"Encontrados {len(items)} arquivos na pasta.")

        # --- Separar arquivos de mídia e documentos existentes ---
        arquivos_midia = {}
        arquivos_ignorados = []
        doc_consolidado_id = None
        doc_consolidado_nome = None

        # Procurar documento consolidado existente conforme o tipo de documento ativo
        prefixo_consolidado = PREFIXO_CONSOLIDADO
        for item in items:
            if (item['mimeType'] == 'application/vnd.google-apps.document'
                    and item['name'].startswith(prefixo_consolidado)):
                doc_consolidado_id = item['id']
                doc_consolidado_nome = item['name']
                print(f"\n📄 Documento consolidado encontrado: '{mascarar(doc_consolidado_nome)}'")
                break

        # Coletar arquivos de mídia
        processed_ids = set()
        for item in items:
            if item['id'] in processed_ids:
                continue
            processed_ids.add(item['id'])

            if not any(item['mimeType'].startswith(mt) for mt in MEDIA_MIME_TYPES):
                arquivos_ignorados.append((item['name'], item['mimeType']))
                continue

            arquivos_midia[item['id']] = {
                'name': item['name'],
                'mimeType': item['mimeType']
            }

        print(f"\nArquivos de mídia encontrados: {len(arquivos_midia)}")
        if arquivos_ignorados:
            print(f"Arquivos ignorados (não multimídia): {len(arquivos_ignorados)}")

        return arquivos_midia, folder_id, doc_consolidado_id

    except Exception as e:
        print(f"Erro ao acessar a pasta: {e}")
        return {}, None, None


def ler_conteudo_doc(document_id):
    """Lê o conteúdo de texto de um Google Doc existente."""
    docs_service = build('docs', 'v1', cache_discovery=False, credentials=creds)
    doc = docs_service.documents().get(documentId=document_id).execute()
    conteudo = ""
    for element in doc.get('body', {}).get('content', []):
        if 'paragraph' in element:
            for run in element['paragraph'].get('elements', []):
                if 'textRun' in run:
                    conteudo += run['textRun']['content']
    return conteudo


def extrair_arquivos_ja_transcritos(conteudo_doc):
    """Analisa o conteúdo do documento consolidado e retorna os nomes dos arquivos já transcritos."""
    ja_transcritos = set()
    # Procura pelo marcador de início de transcrição
    padrao = re.escape(MARCADOR_INICIO) + r"(.+?)\s*═══"
    matches = re.findall(padrao, conteudo_doc)
    for match in matches:
        ja_transcritos.add(match.strip())
    return ja_transcritos


def baixar_arquivo_midia(service, file_id, file_name):
    """Baixa um arquivo de mídia do Google Drive direto para /tmp em blocos de 50 MB.

    Stream para o disco — não carrega o arquivo inteiro em RAM. Suporta com
    folga arquivos de 500 MB a 1 GB+ sem estourar a memória da Colab.
    Retorna o caminho local.
    """
    safe_name = file_name.replace('/', '_')
    local_path = f"/tmp/{safe_name}"
    request = service.files().get_media(fileId=file_id)
    with open(local_path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request, chunksize=50 * 1024 * 1024)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"      ⬇️  {int(status.progress() * 100)}%")
    return local_path


def obter_ou_criar_pasta_audios(folder_id):
    """Obtém ou cria a pasta 'Áudios Extraídos' dentro da pasta do Drive."""
    drive_service = build('drive', 'v3', cache_discovery=False, credentials=creds)
    nome_pasta = "Áudios Extraídos"
    response = drive_service.files().list(
        q=f"'{folder_id}' in parents and name='{nome_pasta}' and mimeType='application/vnd.google-apps.folder' and trashed=false",
        fields='files(id, name)'
    ).execute()
    files = response.get('files', [])
    if files:
        print(f"📁 Pasta '{nome_pasta}' já existe.")
        return files[0]['id']
    file_metadata = {
        'name': nome_pasta,
        'mimeType': 'application/vnd.google-apps.folder',
        'parents': [folder_id]
    }
    file = drive_service.files().create(body=file_metadata, fields='id').execute()
    print(f"📁 Pasta '{nome_pasta}' criada com sucesso.")
    return file.get('id')


def upload_audio_to_drive(local_path, folder_id, file_name):
    """Faz upload de um arquivo de áudio local para uma pasta do Google Drive."""
    drive_service = build('drive', 'v3', cache_discovery=False, credentials=creds)
    file_metadata = {
        'name': file_name,
        'parents': [folder_id]
    }
    media = MediaFileUpload(local_path, mimetype='audio/wav')
    drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()


def mover_arquivo_para_lixeira(file_id, file_name):
    """Move um arquivo do Google Drive para a lixeira (reversível por 30 dias).

    Não é hard-delete — o arquivo continua recuperável via interface do Drive.
    """
    drive_service = build('drive', 'v3', cache_discovery=False, credentials=creds)
    drive_service.files().update(fileId=file_id, body={'trashed': True}).execute()
    print(f"   🗑️  Arquivo original movido para a lixeira: '{mascarar(file_name)}'")


def _segmentos_do_audio(model, path):
    """Transcreve um arquivo de áudio e devolve [(inicio_em_segundos, texto), ...].

    Normaliza o formato dos três motores. Os tempos vêm prontos do modelo — não
    há passo extra de processamento para obtê-los.
    """
    if IS_WHISPERX:
        import whisperx
        resultado = model.transcribe(
            whisperx.load_audio(path), batch_size=WHISPERX_BATCH_SIZE
        )
        return [(s.get("start") or 0.0, s.get("text", "").strip())
                for s in resultado.get("segments", [])]

    if IS_HF_MODEL:
        resultado = model(path, return_timestamps=True)
        trechos = resultado.get("chunks")
        if trechos:
            saida = []
            for t in trechos:
                inicio = (t.get("timestamp") or (None,))[0]
                saida.append((inicio or 0.0, t.get("text", "").strip()))
            return saida
        return [(0.0, resultado.get("text", "").strip())]

    resultado = model.transcribe(path, language="pt")
    segmentos = resultado.get("segments")
    if segmentos:
        return [(s.get("start") or 0.0, s.get("text", "").strip()) for s in segmentos]
    return [(0.0, resultado.get("text", "").strip())]


def formatar_tempo(segundos):
    """Segundos → 'HH:MM:SS'."""
    segundos = int(segundos)
    return f"{segundos // 3600:02d}:{(segundos % 3600) // 60:02d}:{segundos % 60:02d}"


def formatar_transcricao(segmentos):
    """Monta o texto final a partir de [(inicio, texto), ...].

    Conforme INTERVALO_CARIMBO_S: agrupa em blocos de duração fixa carimbados
    pelo início do bloco (0), carimba cada segmento do modelo (0) ou devolve
    texto corrido (None). Devolve None se não houver texto.

    O carimbo do bloco é o instante cheio (00:03:00), não o começo do primeiro
    trecho dentro dele — assim serve para localizar direto no áudio.
    """
    segmentos = [(t, txt) for t, txt in segmentos if txt]
    if not segmentos:
        return None

    if INTERVALO_CARIMBO_S is None:
        return " ".join(txt for _, txt in segmentos).strip()

    if INTERVALO_CARIMBO_S == 0:
        return "\n".join(f"[{formatar_tempo(t)}] {txt}" for t, txt in segmentos).strip()

    blocos = []
    for inicio, txt in segmentos:
        marca = int(inicio // INTERVALO_CARIMBO_S) * INTERVALO_CARIMBO_S
        if not blocos or blocos[-1][0] != marca:
            blocos.append((marca, []))
        blocos[-1][1].append(txt)

    return "\n\n".join(
        f"[{formatar_tempo(marca)}] {' '.join(partes)}" for marca, partes in blocos
    ).strip()


def transcrever_arquivo(model, file_info, guardar_audio=False):
    """Transcreve um único arquivo de mídia já baixado em disco.

    Espera um dict com chaves: 'name', 'path' (caminho local em /tmp) e 'mimeType'.
    Retorna (transcrição, caminho_audio_extraido ou None).

    Se a duração do arquivo ultrapassar LIMITE_DURACAO_S, o áudio é extraído para
    WAV, fragmentado em pedaços de CHUNK_DURACAO_S segundos via ffmpeg, e cada
    pedaço é transcrito separadamente — as partes são concatenadas no final, com
    os tempos deslocados para continuarem coerentes com o arquivo inteiro.
    Combinado com o download streaming, evita estouros de memória em arquivos
    de 500 MB a 1 GB+.
    """
    file_path = file_info['path']

    audio_extraido_path = None
    chunks_temp = []
    try:
        is_video = file_info['mimeType'].startswith('video/')
        duracao = obter_duracao_segundos(file_path)

        if duracao > LIMITE_DURACAO_S:
            # Caminho com fragmentação para arquivos longos
            print(f"   ✂️  Arquivo longo ({duracao/60:.1f} min). Fragmentando em pedaços de {CHUNK_DURACAO_S/60:.0f} min...")
            audio_consolidado = extract_audio(file_path)
            chunks_temp = fragmentar_audio(audio_consolidado, CHUNK_DURACAO_S)
            print(f"   📦 {len(chunks_temp)} fragmento(s) gerado(s).")

            # Os tempos que cada fragmento devolve começam do zero; somamos o
            # deslocamento do fragmento para que o carimbo se refira ao arquivo
            # inteiro, e não ao pedaço.
            segmentos = []
            for idx, chunk_path in enumerate(chunks_temp):
                print(f"   🎤 Transcrevendo fragmento {idx + 1}/{len(chunks_temp)}...")
                deslocamento = idx * CHUNK_DURACAO_S
                segmentos += [(inicio + deslocamento, txt)
                              for inicio, txt in _segmentos_do_audio(model, chunk_path)]
            transcription = formatar_transcricao(segmentos)

            if guardar_audio and is_video:
                audio_extraido_path = audio_consolidado
            else:
                if os.path.exists(audio_consolidado):
                    os.remove(audio_consolidado)
        else:
            # Caminho original (sem fragmentação) para arquivos curtos
            if IS_HF_MODEL:
                audio_path = extract_audio(file_path)
                transcription = formatar_transcricao(
                    _segmentos_do_audio(model, audio_path)
                )
                if guardar_audio and is_video:
                    audio_extraido_path = audio_path
            else:
                transcription = formatar_transcricao(
                    _segmentos_do_audio(model, file_path)
                )
                if guardar_audio and is_video:
                    audio_extraido_path = extract_audio(file_path)

        return (transcription if transcription else None), audio_extraido_path
    except Exception as e:
        print(f"❌ Erro ao transcrever {mascarar(file_info['name'])}: {e}")
        return None, None
    finally:
        # Limpar arquivo original temporário
        if os.path.exists(file_path):
            os.remove(file_path)
        # Limpar fragmentos temporários
        for c in chunks_temp:
            if os.path.exists(c):
                os.remove(c)
        # Limpar wav temporário apenas se não for guardar
        wav_path = file_path.rsplit(".", 1)[0] + ".wav"
        if not audio_extraido_path and os.path.exists(wav_path):
            os.remove(wav_path)


def criar_google_doc(folder_id, title):
    """Cria um documento Google Docs na pasta especificada."""
    drive_service = build('drive', 'v3', cache_discovery=False, credentials=creds)
    file_metadata = {
        'name': title,
        'mimeType': 'application/vnd.google-apps.document',
        'parents': [folder_id]
    }
    file = drive_service.files().create(body=file_metadata, fields='id').execute()
    file_id = file.get('id')
    print(f"\n📄 Documento '{mascarar(title)}' criado com sucesso.")
    return file_id


def obter_indice_para_proxima_transcricao(document_id):
    """Calcula o índice (Docs API) para inserir a próxima transcrição.

    A próxima transcrição vai logo após a última existente.
    """
    docs_service = build('docs', 'v1', cache_discovery=False, credentials=creds)
    doc = docs_service.documents().get(documentId=document_id).execute()
    full_text = ""
    for element in doc.get('body', {}).get('content', []):
        if 'paragraph' in element:
            for run in element['paragraph'].get('elements', []):
                if 'textRun' in run:
                    full_text += run['textRun']['content']

    # Caso 1: já existem transcrições → inserir após o último separador "─" * 60
    ultimo_fim = full_text.rfind(MARCADOR_FIM)
    if ultimo_fim != -1:
        separador = "─" * 60
        sep_pos = full_text.find(separador, ultimo_fim)
        if sep_pos != -1:
            end_idx = sep_pos + len(separador)
            if end_idx < len(full_text) and full_text[end_idx] == "\n":
                end_idx += 1
            return end_idx + 1  # +1 → índice da Docs API

    # Caso 2: sem transcrições mas com sumário → inserir após fim do sumário
    if MARCADOR_SUMARIO in full_text:
        marca_fim = "═" * 60
        pos = full_text.rfind(marca_fim)
        if pos != -1:
            end_idx = pos + len(marca_fim)
            if end_idx < len(full_text) and full_text[end_idx] == "\n":
                end_idx += 1
            return end_idx + 1

    # Caso 3: fallback — fim do documento
    body = doc.get('body', {})
    content = body.get('content', [])
    if content:
        return content[-1].get('endIndex', 1) - 1
    return 1


def inserir_conteudo_no_doc(document_id, content, index=1):
    """Insere conteúdo no documento Google Docs numa posição específica."""
    docs_service = build('docs', 'v1', cache_discovery=False, credentials=creds)
    requests = [
        {
            'insertText': {
                'location': {'index': index},
                'text': content
            }
        }
    ]
    docs_service.documents().batchUpdate(
        documentId=document_id,
        body={'requests': requests}
    ).execute()


def substituir_sumario(document_id, novo_sumario_texto):
    """Substitui o sumário no documento. Lê o doc, encontra a seção do sumário e a substitui."""
    docs_service = build('docs', 'v1', cache_discovery=False, credentials=creds)
    doc = docs_service.documents().get(documentId=document_id).execute()

    # Reconstruir o texto completo com índices
    full_text = ""
    for element in doc.get('body', {}).get('content', []):
        if 'paragraph' in element:
            for run in element['paragraph'].get('elements', []):
                if 'textRun' in run:
                    full_text += run['textRun']['content']

    # Encontrar onde o sumário começa e termina
    inicio_sumario = full_text.find(MARCADOR_SUMARIO)
    if inicio_sumario == -1:
        return  # Sem sumário para substituir

    # O sumário vai do marcador até a primeira transcrição (ou até o próximo marcador)
    inicio_primeira_transcricao = full_text.find(MARCADOR_INICIO)
    if inicio_primeira_transcricao == -1:
        fim_sumario = len(full_text)
    else:
        fim_sumario = inicio_primeira_transcricao

    # Precisamos dos índices no documento (offset +1 por causa do índice do Google Docs)
    # Nota: o índice do Google Docs começa em 1
    doc_inicio = inicio_sumario + 1
    doc_fim = fim_sumario + 1

    requests = [
        {
            'deleteContentRange': {
                'range': {
                    'startIndex': doc_inicio,
                    'endIndex': doc_fim,
                }
            }
        },
        {
            'insertText': {
                'location': {'index': doc_inicio},
                'text': novo_sumario_texto
            }
        }
    ]
    docs_service.documents().batchUpdate(
        documentId=document_id,
        body={'requests': requests}
    ).execute()


def montar_texto_sumario(lista_arquivos, ja_transcritos_set):
    """Monta o texto do sumário com a lista de arquivos e status."""
    linhas = [f"{MARCADOR_SUMARIO}\n\n"]
    for i, nome in enumerate(lista_arquivos, 1):
        status = "✅" if nome in ja_transcritos_set else "⏳"
        linhas.append(f"{i}. {status} {nome}\n")
    linhas.append(f"\nTotal: {len(lista_arquivos)} arquivo(s)\n")
    linhas.append(f"Transcritos: {len(ja_transcritos_set)} | Pendentes: {len(lista_arquivos) - len(ja_transcritos_set)}\n")
    linhas.append("\n" + "═" * 60 + "\n\n")
    return "".join(linhas)


def montar_bloco_transcricao(nome_arquivo, transcricao):
    """Monta o bloco de texto de uma transcrição individual."""
    bloco = (
        f"\n{MARCADOR_INICIO}{nome_arquivo} ═══\n"
        f"\n{transcricao}\n\n"
        f"{MARCADOR_FIM}{nome_arquivo} ═══\n"
        f"\n{'─' * 60}\n"
    )
    return bloco


# ════════════════════════════════════════════════════════════════
# FUNÇÃO PRINCIPAL
# ════════════════════════════════════════════════════════════════

def obter_arquivos_via_upload():
    """Abre o seletor de upload da Colab e grava os arquivos de mídia em /tmp.

    Retorna uma lista de dicts {name, path, mimeType}, ignorando arquivos que
    não sejam de áudio ou vídeo. Tudo permanece no disco temporário do Colab.
    """
    from google.colab import files
    import mimetypes
    EXT_VIDEO = {"mp4", "mov", "avi", "mkv", "webm", "flv", "wmv", "m4v",
                 "mpeg", "mpg", "3gp", "ts"}
    EXT_AUDIO = {"mp3", "wav", "m4a", "aac", "ogg", "oga", "flac", "wma",
                 "opus", "amr", "aiff"}
    print("📤 Selecione o(s) arquivo(s) de áudio ou vídeo do seu computador...")
    uploaded = files.upload()
    if not uploaded:
        print("⚠️ Nenhum arquivo enviado.")
        return []
    itens = []
    for nome, conteudo in uploaded.items():
        ext = nome.rsplit(".", 1)[-1].lower() if "." in nome else ""
        mime, _ = mimetypes.guess_type(nome)
        if mime and (mime.startswith("video/") or mime.startswith("audio/")):
            tipo = mime
        elif ext in EXT_VIDEO:
            tipo = "video/" + ext
        elif ext in EXT_AUDIO:
            tipo = "audio/" + ext
        else:
            print(f"   ⚠️ Ignorado (não é áudio nem vídeo): {mascarar(nome)}")
            continue
        safe = nome.replace("/", "_")
        path = f"/tmp/{safe}"
        with open(path, "wb") as fh:
            fh.write(conteudo)
        itens.append({"name": nome, "path": path, "mimeType": tipo})
        print(f"   ✅ Recebido: {mascarar(nome)} ({len(conteudo) / 1e6:.1f} MB)")
    return itens


def transcrever_uploads():
    """Fluxo do modo upload: transcreve arquivos enviados do computador e
    devolve um .txt consolidado para download — sem usar o Google Drive.

    O sumário e os blocos de transcrição seguem o mesmo formato do documento
    consolidado do modo Drive.
    """
    itens = obter_arquivos_via_upload()
    if not itens:
        print("⚠️ Nenhum arquivo de áudio ou vídeo para transcrever. Encerrando.")
        return None

    itens.sort(key=lambda x: x["name"])
    nomes_arquivos = [it["name"] for it in itens]

    print(f"\n📋 Arquivos a transcrever ({len(itens)}):")
    for it in itens:
        print(f"   ⏳ {mascarar(it['name'])}")

    transcritos = set()
    blocos = []
    for i, it in enumerate(itens):
        nome = it["name"]
        print(f"\n{'='*60}")
        print(f"🎤 [{i+1}/{len(itens)}] Transcrevendo: {mascarar(nome)}")
        print(f"{'='*60}")

        start_time = time.time()
        # guardar_audio=False: no modo upload não há Drive para salvar o áudio extraído.
        transcricao, _ = transcrever_arquivo(model, it, guardar_audio=False)
        elapsed = time.time() - start_time

        if not transcricao:
            print(f"   ❌ Transcrição vazia ou com erro para {mascarar(nome)}. Pulando...")
            continue

        print(f"   ✅ Transcrição concluída em {elapsed:.1f}s ({len(transcricao)} caracteres)")
        transcritos.add(nome)
        blocos.append(montar_bloco_transcricao(nome, transcricao))

    if not blocos:
        print("\n⚠️ Nenhuma transcrição foi gerada. Nada para baixar.")
        return None

    # Monta o documento consolidado em texto (sumário + blocos de transcrição)
    sumario = montar_texto_sumario(nomes_arquivos, transcritos)
    conteudo = sumario + "".join(blocos)

    primeiro_nome = os.path.splitext(nomes_arquivos[0])[0]
    sufixo = f"{primeiro_nome} e outros" if len(nomes_arquivos) > 1 else primeiro_nome
    nome_saida = f"{PREFIXO_CONSOLIDADO} {sufixo}.txt".replace("/", "-")
    caminho_saida = f"/tmp/{nome_saida}"
    with open(caminho_saida, "w", encoding="utf-8") as fh:
        fh.write(conteudo)

    # Limpa todo o log da execução antes de disparar o download. O que ficar
    # salvo no notebook é só o resumo abaixo — sem nomes de arquivos.
    limpar_log()
    print(f"{'='*60}")
    print(f"🎉 Processamento concluído!")
    print(f"   Transcritos: {len(transcritos)}/{len(nomes_arquivos)}")
    print(f"   💾 Baixando o arquivo de transcrições para o seu computador.")
    print(f"{'='*60}")

    from google.colab import files
    files.download(caminho_saida)
    return [caminho_saida]


def generate_transcriptions():
    # Sem link de pasta: cai no modo upload — processa no disco temporário do
    # Colab e baixa o .txt no fim, sem tocar no Google Drive.
    if not USAR_DRIVE:
        return transcrever_uploads()

    # 1. Listar arquivos da pasta
    arquivos_midia, folder_id, doc_consolidado_id = get_documents_from_drive(PASTA_DOCUMENTOS)

    if not arquivos_midia:
        print("⚠️ Nenhum arquivo de áudio ou vídeo encontrado na pasta. Encerrando.")
        return None

    # Ordenar por nome para consistência
    lista_ordenada = sorted(arquivos_midia.items(), key=lambda x: x[1]['name'])
    nomes_arquivos = [info['name'] for _, info in lista_ordenada]

    # 2. Verificar se já existe documento consolidado e quais arquivos já foram transcritos
    ja_transcritos = set()
    if doc_consolidado_id:
        print("\n🔍 Lendo documento existente para verificar transcrições já feitas...")
        conteudo_existente = ler_conteudo_doc(doc_consolidado_id)
        ja_transcritos = extrair_arquivos_ja_transcritos(conteudo_existente)
        if ja_transcritos:
            print(f"\n✅ Arquivos já transcritos ({len(ja_transcritos)}):")
            for nome in ja_transcritos:
                print(f"   • {mascarar(nome)}")

    # 3. Determinar quais arquivos ainda precisam ser transcritos
    pendentes = [(fid, info) for fid, info in lista_ordenada if info['name'] not in ja_transcritos]

    if not pendentes:
        if doc_consolidado_id:
            link = f"https://docs.google.com/document/d/{doc_consolidado_id}/edit"
            limpar_log()
            print("🎉 Todos os arquivos já foram transcritos! Nada a fazer.")
            print(f"   📄 Documento: {link}")
            return [link]
        limpar_log()
        print("🎉 Todos os arquivos já foram transcritos! Nada a fazer.")
        return None

    print(f"\n📋 Arquivos pendentes de transcrição ({len(pendentes)}):")
    for _, info in pendentes:
        print(f"   ⏳ {mascarar(info['name'])}")

    # 4. Criar ou reutilizar o documento consolidado
    if not doc_consolidado_id:
        primeiro_nome = nomes_arquivos[0]
        sufixo = f"{primeiro_nome} e outros" if len(nomes_arquivos) > 1 else primeiro_nome

        doc_title = f"{PREFIXO_CONSOLIDADO} {sufixo}"
        doc_consolidado_id = criar_google_doc(folder_id, doc_title)
        sumario_texto = montar_texto_sumario(nomes_arquivos, ja_transcritos)
        inserir_conteudo_no_doc(doc_consolidado_id, sumario_texto, index=1)
        print("📑 Sumário inicial inserido no documento.")

    # 5. Transcrever arquivos pendentes e adicionar ao documento
    drive_service = build('drive', 'v3', cache_discovery=False, credentials=creds)

    # Criar pasta para áudios extraídos se configurado
    pasta_audios_id = None
    if GUARDAR_AUDIOS:
        pasta_audios_id = obter_ou_criar_pasta_audios(folder_id)

    for i, (file_id, file_info) in enumerate(pendentes):
        nome = file_info['name']
        print(f"\n{'='*60}")
        print(f"🎤 [{i+1}/{len(pendentes)}] Transcrevendo: {mascarar(nome)}")
        print(f"{'='*60}")

        # Baixar o arquivo (streaming direto pro disco — sem segurar tudo em RAM)
        try:
            print(f"   ⬇️  Baixando arquivo...")
            caminho_local = baixar_arquivo_midia(drive_service, file_id, nome)
        except Exception as e:
            print(f"   ❌ Erro ao baixar {mascarar(nome)}: {e}")
            continue

        file_info_baixado = {
            'name': nome,
            'path': caminho_local,
            'mimeType': file_info['mimeType']
        }

        # Transcrever
        start_time = time.time()
        transcricao, audio_extraido_path = transcrever_arquivo(model, file_info_baixado, guardar_audio=GUARDAR_AUDIOS)
        elapsed = time.time() - start_time

        if not transcricao:
            print(f"   ❌ Transcrição vazia ou com erro para {mascarar(nome)}. Pulando...")
            # Limpar áudio extraído residual se houver
            if audio_extraido_path and os.path.exists(audio_extraido_path):
                os.remove(audio_extraido_path)
            continue

        print(f"   ✅ Transcrição concluída em {elapsed:.1f}s ({len(transcricao)} caracteres)")

        # Upload do áudio extraído se configurado
        if audio_extraido_path and pasta_audios_id:
            try:
                nome_audio = os.path.splitext(nome)[0] + ".wav"
                upload_audio_to_drive(audio_extraido_path, pasta_audios_id, nome_audio)
                print(f"   🔊 Áudio extraído salvo no Drive: '{mascarar(nome_audio)}'")
            except Exception as e:
                print(f"   ⚠️ Erro ao salvar áudio extraído: {e}")
            finally:
                if os.path.exists(audio_extraido_path):
                    os.remove(audio_extraido_path)

        # Montar bloco e inserir após a última transcrição
        bloco = montar_bloco_transcricao(nome, transcricao)
        try:
            indice = obter_indice_para_proxima_transcricao(doc_consolidado_id)
            inserir_conteudo_no_doc(doc_consolidado_id, bloco, index=indice)
            print(f"   📝 Transcrição adicionada ao documento consolidado.")
        except Exception as e:
            print(f"   ❌ Erro ao inserir transcrição no documento: {e}")
            continue

        # Atualizar set de transcritos
        ja_transcritos.add(nome)

        # Atualizar o sumário no documento
        try:
            novo_sumario = montar_texto_sumario(nomes_arquivos, ja_transcritos)
            substituir_sumario(doc_consolidado_id, novo_sumario)
            print(f"   📑 Sumário atualizado.")
        except Exception as e:
            print(f"   ⚠️ Não foi possível atualizar o sumário: {e}")

        # Apagar o arquivo original (mover para lixeira) se solicitado
        if APAGAR_ORIGINAL:
            try:
                mover_arquivo_para_lixeira(file_id, nome)
            except Exception as e:
                print(f"   ⚠️ Não foi possível mover '{mascarar(nome)}' para a lixeira: {e}")

    # 6. Resultado final
    link = f"https://docs.google.com/document/d/{doc_consolidado_id}/edit"
    # Limpa todo o log da execução: o que sobra (e fica salvo no notebook) é
    # apenas o resumo abaixo, com o link do documento.
    limpar_log()
    print(f"{'='*60}")
    print(f"🎉 Processamento concluído!")
    print(f"   Transcritos: {len(ja_transcritos)}/{len(nomes_arquivos)}")
    print(f"   📄 Documento: {link}")
    print(f"{'='*60}")
    return [link]


# ════════════════════════════════════════════════════════════════
# EXECUÇÃO
# ════════════════════════════════════════════════════════════════

resultado = generate_transcriptions()
if resultado:
    print("\n✅ Todas as transcrições foram salvas no documento consolidado!")
else:
    print("\nHouve um erro ou não havia arquivos para transcrever.")